## Claude.ai

Push VQA to Claude.

### Docs:

* in theory, there is documentation here: https://docs.anthropic.com/en/home, 
* but in reality I just asked "Hey Claude.  How do I use your API through Python to upload an image and ask you questions about it?" followed by "Is there anyway to set the context?  For example, in chatgpt you have the ""role": "system", "content"" part of your message where you would say things like "You are a helpful assistant in charge of automating a process".  Or does one just incorporate that into the "question" part of the inputs?" and starting building from there

#### Key Points (cp-claude)

* API Key: Get your API key from the [Anthropic Console](https://console.anthropic.com/) and set it as an environment variable or pass it directly
* Supported formats: JPEG, PNG, GIF, and WebP images
* Model: Use `claude-sonnet-4-20250514` for Claude Sonnet 4
* Size limits: Images should be under 5MB and no larger than 8000x8000 pixels



First, install anthropic api (also, see .yml file for the environment for this project)

In [1]:
# !pip install anthropic

Where are things stored/going to be stored?

In [2]:
# ============================ RUN CONFIG ============================
# LOW-TIER run.  "Low tier" is the no-reasoning / no-thinking configuration --
# the baseline the paper's `low_dirs_to_use` used (chatgpt_api, gemini,
# claude_haiku).  The mid/high-tier variants are kept commented out below.
#
# Model ID note: Claude model IDs are now used WITHOUT a date suffix --
# "claude-haiku-4-5", not "claude-haiku-4-5-20251001".  Haiku 4.5 is still the
# current low-tier Claude model (Sonnet 5 / Opus 5 are the higher tiers).
out_base = '~/astro_sky_image_vqa/LMM_outputs_n150_archive_light/'   # ARCHIVE-AGED images   # TEST tree, not the real one

dir_api = out_base + 'claude_haiku/'   # low tier: no thinking
model = "claude-haiku-4-5"             # nano-like equivalent
thinking = False
max_tokens = 2000                      # response only; no thinking budget needed

# ---- mid/high tier (previous runs), for reference ----
# dir_api = out_base + 'claude_haiku_thinking_maxT8000/'
# model = "claude-haiku-4-5"
# thinking = True
# max_tokens = 10000   # 8000 (thinking) + 2000 (response)
#
# dir_api = out_base + 'claude_sonnet/'
# model = "claude-sonnet-5"   # 4.6 -> 5 is the current Sonnet
# thinking = True
# max_tokens = 10000

# where is VQA dataset?
# TODO: these still point at the old jcdl_followup set -- repoint once the
# WASP2026 VQA questions have been generated.
jsons_dir = '~/astro_sky_image_vqa/VQA_full/qa_jsons/' # directory where jsons created with figure are stored
imgs_dir = '~/astro_sky_image_vqa/VQA_full/imgs/' # where images are stored

# other stuff, where stored?
key_file = '/Users/jnaiman/.claudeai/key.txt'
# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/'

img_format = 'jpeg'

# ---- archive aging (LIGHT) --------------------------------------------------
# Every figure is aged with the "archive-light" preset before it is sent (see
# the ARCHIVE AGING cell below): the same journey as "archive" -- stored for
# decades, then scanned -- but every effect softened and firing less often, so
# the figure stays legible.  Same presets as page_aging.ipynb.
archive_gray_prob = 0.6667     # fraction of figures dropped to grayscale (0.0 - 1.0)
archive_base_seed = 20260914   # change this to re-roll the whole set
archive_preset    = 'archive-light'   # 'archive' is the full-strength version

# Colour augraphy uses for anything it has to invent -- the wedges a rotation
# leaves at the corners, the backdrop of a fold.  It defaults to BLACK, which
# on a scanned page reads as a hole rather than as paper.  BGR.
archive_page_background = (255, 255, 255)

# The draw is seeded per figure from its id plus archive_base_seed, so all three
# model notebooks see the SAME aged image for a given figure -- change
# archive_base_seed in ALL of them together, or they will diverge.

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'


In [3]:
import anthropic
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob

# debug
from importlib import reload
from anthropic import RateLimitError, APIStatusError, APIConnectionError, BadRequestError


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
import utils.archive_aging
reload(utils.archive_aging)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

# --- expand ~ ONCE, and assign back ----------------------------------------
# These must be rebound, not expanded locally: everything downstream (glob,
# open, os.path.join) uses the variables directly, and glob('~/...') silently
# matches nothing rather than erroring.
out_base  = os.path.expanduser(out_base)
dir_api   = os.path.expanduser(dir_api)
key_file  = os.path.expanduser(key_file)
jsons_dir = os.path.expanduser(jsons_dir)
imgs_dir  = os.path.expanduser(imgs_dir)
tmp_dir   = os.path.expanduser(tmp_dir)

# --- make the directories we WRITE to -------------------------------------
# dir_api holds one <id>_qa.pickle per figure; tmp_dir holds the resized images
# load_image() falls back to when a request is too large.  Neither is created
# by anything upstream, so a fresh out_base fails on the first save.
for _d in (dir_api, tmp_dir):
    if not os.path.exists(_d):
        os.makedirs(_d, exist_ok=True)   # makedirs, not mkdir: the path is nested
        print('made:', _d)

# --- check the directories we READ from ------------------------------------
for _name, _d in (('jsons_dir', jsons_dir), ('imgs_dir', imgs_dir)):
    if not os.path.isdir(_d):
        print('[WARN] %s does not exist: %s' % (_name, _d))
if not os.path.exists(key_file):
    print('[WARN] key_file does not exist:', key_file)


made: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/


In [4]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

client = anthropic.Anthropic(
  api_key=api_key.strip(),  # this is also the default, it can be omitted
)

In [5]:
# ================= QUICK TEST: five figures per category =================
# The released VQA ids are shuffled and carry no category (deliberately -- the
# generator's own names encoded the real-vs-synthetic answer). So classify by
# reading each qa json instead of by filename.
#
# TEST_IDS pins the exact figures so the test is reproducible, and is the SAME
# set used by the other test notebooks, so the models are comparable. Each
# value may be a single id or a list of them. Set TEST_IDS = None to auto-pick
# the first N_PER_CATEGORY of each category and print a fresh block.

# ---- the original one-per-category run; uncomment this block (and comment out
# ---- the five-per-category one below) to go back to a 3-figure test ----
# TEST_IDS = {
#     'contour':  'vqa_000003',
#     'sky-real': 'vqa_000001',
#     'sky-gmm':  'vqa_000004',
# }

# ---- the five-per-category run; uncomment to go back to a 15-figure test ----
# TEST_IDS = {
#     'contour':  ['vqa_000003', 'vqa_000006', 'vqa_000010', 'vqa_000012', 'vqa_000014'],
#     'sky-real': ['vqa_000001', 'vqa_000002', 'vqa_000009', 'vqa_000021', 'vqa_000022'],
#     'sky-gmm':  ['vqa_000004', 'vqa_000005', 'vqa_000007', 'vqa_000008', 'vqa_000011'],
# }

# Fifty per category = 150 figures.  These are DISJOINT from every id used by the
# 3- and 15-figure runs above, so no figure is scored twice across the test sets;
# they are the first 50 of each category, in sorted id order, skipping those.
# ---- the previous fifty-per-category run.  Those ids came from the SUPERSEDED
# ---- lookup table (archived as vqa_lookup_superseded_*.json): the dataset was
# ---- regenerated afterwards and every id was reassigned, so they no longer
# ---- point at the figures they were scored on.  Kept for reference only.
# TEST_IDS = {
#     'contour':  ['vqa_000016', 'vqa_000017', 'vqa_000018', 'vqa_000019', 'vqa_000020',
#                  'vqa_000025', 'vqa_000029', 'vqa_000032', 'vqa_000033', 'vqa_000035',
#                  'vqa_000038', 'vqa_000039', 'vqa_000041', 'vqa_000043', 'vqa_000045',
#                  'vqa_000049', 'vqa_000050', 'vqa_000053', 'vqa_000055', 'vqa_000056',
#                  'vqa_000064', 'vqa_000066', 'vqa_000068', 'vqa_000069', 'vqa_000070',
#                  'vqa_000072', 'vqa_000084', 'vqa_000085', 'vqa_000088', 'vqa_000091',
#                  'vqa_000097', 'vqa_000103', 'vqa_000105', 'vqa_000106', 'vqa_000111',
#                  'vqa_000112', 'vqa_000113', 'vqa_000118', 'vqa_000121', 'vqa_000124',
#                  'vqa_000135', 'vqa_000141', 'vqa_000143', 'vqa_000144', 'vqa_000146',
#                  'vqa_000151', 'vqa_000154', 'vqa_000155', 'vqa_000156', 'vqa_000158'],
#     'sky-real': ['vqa_000028', 'vqa_000036', 'vqa_000037', 'vqa_000040', 'vqa_000042',
#                  'vqa_000044', 'vqa_000047', 'vqa_000048', 'vqa_000052', 'vqa_000054',
#                  'vqa_000057', 'vqa_000059', 'vqa_000060', 'vqa_000063', 'vqa_000074',
#                  'vqa_000076', 'vqa_000078', 'vqa_000079', 'vqa_000082', 'vqa_000083',
#                  'vqa_000087', 'vqa_000092', 'vqa_000093', 'vqa_000095', 'vqa_000096',
#                  'vqa_000098', 'vqa_000101', 'vqa_000102', 'vqa_000107', 'vqa_000109',
#                  'vqa_000114', 'vqa_000115', 'vqa_000116', 'vqa_000117', 'vqa_000120',
#                  'vqa_000125', 'vqa_000126', 'vqa_000128', 'vqa_000130', 'vqa_000131',
#                  'vqa_000132', 'vqa_000133', 'vqa_000134', 'vqa_000137', 'vqa_000138',
#                  'vqa_000140', 'vqa_000149', 'vqa_000150', 'vqa_000152', 'vqa_000157'],
#     'sky-gmm':  ['vqa_000013', 'vqa_000015', 'vqa_000023', 'vqa_000024', 'vqa_000026',
#                  'vqa_000027', 'vqa_000030', 'vqa_000031', 'vqa_000034', 'vqa_000046',
#                  'vqa_000051', 'vqa_000058', 'vqa_000061', 'vqa_000062', 'vqa_000065',
#                  'vqa_000067', 'vqa_000071', 'vqa_000073', 'vqa_000075', 'vqa_000077',
#                  'vqa_000080', 'vqa_000081', 'vqa_000086', 'vqa_000089', 'vqa_000090',
#                  'vqa_000094', 'vqa_000099', 'vqa_000100', 'vqa_000104', 'vqa_000108',
#                  'vqa_000110', 'vqa_000119', 'vqa_000122', 'vqa_000123', 'vqa_000127',
#                  'vqa_000129', 'vqa_000136', 'vqa_000139', 'vqa_000142', 'vqa_000145',
#                  'vqa_000147', 'vqa_000148', 'vqa_000153', 'vqa_000159', 'vqa_000163',
#                  'vqa_000164', 'vqa_000166', 'vqa_000167', 'vqa_000169', 'vqa_000170'],
# }

# Fifty per category = 150 figures, drawn from the REBUILT lookup table (the one
# covering the regenerated dataset: TESS excluded, real fields given a deliberate
# angular size, gmm field sizes and colour scales matched to the real family).
# Taken as the first 50 of each category in sorted id order -- the ids are
# already a single-pass shuffle of the source figures, so that is an unbiased
# sample and needs no separate seed.  The SAME 150 are used by every test
# notebook, so the three models and the three aging levels stay comparable.
TEST_IDS = {
    'contour':  ['vqa_000007', 'vqa_000010', 'vqa_000011', 'vqa_000013', 'vqa_000017',
                 'vqa_000023', 'vqa_000024', 'vqa_000025', 'vqa_000030', 'vqa_000032',
                 'vqa_000034', 'vqa_000035', 'vqa_000038', 'vqa_000040', 'vqa_000041',
                 'vqa_000042', 'vqa_000052', 'vqa_000057', 'vqa_000061', 'vqa_000063',
                 'vqa_000065', 'vqa_000067', 'vqa_000070', 'vqa_000072', 'vqa_000077',
                 'vqa_000078', 'vqa_000087', 'vqa_000088', 'vqa_000089', 'vqa_000090',
                 'vqa_000092', 'vqa_000094', 'vqa_000095', 'vqa_000097', 'vqa_000101',
                 'vqa_000102', 'vqa_000103', 'vqa_000105', 'vqa_000107', 'vqa_000109',
                 'vqa_000110', 'vqa_000111', 'vqa_000115', 'vqa_000127', 'vqa_000128',
                 'vqa_000132', 'vqa_000136', 'vqa_000140', 'vqa_000141', 'vqa_000146'],
    'sky-real': ['vqa_000002', 'vqa_000003', 'vqa_000004', 'vqa_000005', 'vqa_000006',
                 'vqa_000008', 'vqa_000009', 'vqa_000015', 'vqa_000018', 'vqa_000019',
                 'vqa_000020', 'vqa_000027', 'vqa_000028', 'vqa_000029', 'vqa_000031',
                 'vqa_000033', 'vqa_000036', 'vqa_000044', 'vqa_000045', 'vqa_000046',
                 'vqa_000049', 'vqa_000050', 'vqa_000055', 'vqa_000066', 'vqa_000074',
                 'vqa_000080', 'vqa_000082', 'vqa_000086', 'vqa_000091', 'vqa_000093',
                 'vqa_000096', 'vqa_000098', 'vqa_000099', 'vqa_000100', 'vqa_000104',
                 'vqa_000106', 'vqa_000108', 'vqa_000117', 'vqa_000118', 'vqa_000120',
                 'vqa_000121', 'vqa_000122', 'vqa_000124', 'vqa_000125', 'vqa_000126',
                 'vqa_000129', 'vqa_000133', 'vqa_000137', 'vqa_000138', 'vqa_000139'],
    'sky-gmm':  ['vqa_000001', 'vqa_000012', 'vqa_000014', 'vqa_000016', 'vqa_000021',
                 'vqa_000022', 'vqa_000026', 'vqa_000037', 'vqa_000039', 'vqa_000043',
                 'vqa_000047', 'vqa_000048', 'vqa_000051', 'vqa_000053', 'vqa_000054',
                 'vqa_000056', 'vqa_000058', 'vqa_000059', 'vqa_000060', 'vqa_000062',
                 'vqa_000064', 'vqa_000068', 'vqa_000069', 'vqa_000071', 'vqa_000073',
                 'vqa_000075', 'vqa_000076', 'vqa_000079', 'vqa_000081', 'vqa_000083',
                 'vqa_000084', 'vqa_000085', 'vqa_000112', 'vqa_000113', 'vqa_000114',
                 'vqa_000116', 'vqa_000119', 'vqa_000123', 'vqa_000130', 'vqa_000131',
                 'vqa_000134', 'vqa_000135', 'vqa_000142', 'vqa_000143', 'vqa_000144',
                 'vqa_000145', 'vqa_000149', 'vqa_000151', 'vqa_000156', 'vqa_000157'],
}
# TEST_IDS = None   # auto-pick instead

CATEGORIES = ['contour', 'sky-real', 'sky-gmm']
N_PER_CATEGORY = 50       # only consulted when TEST_IDS is None


def categorise_qa_json(qa_json_path):
    """contour / sky-real / sky-gmm, from the panel types + distributions."""
    with open(qa_json_path, 'r') as f:
        d = json.loads(json.load(f))
    types, dists = set(), set()
    for k, v in d.items():
        if k.startswith('plot'):
            types.add(v.get('type'))
            dists.add(v.get('distribution'))
    if types == {'contour'}:
        return 'contour'
    if types == {'image of the sky'}:
        if dists == {'sky'}:
            return 'sky-real'
        if dists == {'gmm'}:
            return 'sky-gmm'
        return 'sky-mixed'
    return 'other'


def has_image(qa_json_path):
    stem = os.path.basename(qa_json_path).removesuffix('_qa.json')
    return os.path.exists(os.path.join(imgs_dir, stem + '.' + img_format))


all_qa = sorted(glob(os.path.join(jsons_dir, '*.json')))
all_qa = [j for j in all_qa if has_image(j)]
print('total qa files with a matching image:', len(all_qa))

jsons_to_parse = []
if TEST_IDS:
    for cat in CATEGORIES:
        vids = TEST_IDS.get(cat) or []
        if isinstance(vids, str):        # a bare id, as the 3-figure block uses
            vids = [vids]
        for vid in vids:
            pth = os.path.join(jsons_dir, vid + '_qa.json')
            if not os.path.exists(pth):
                print('[WARN] %s (%s) not found -- skipping' % (vid, cat))
                continue
            jsons_to_parse.append(pth)
else:
    # first N_PER_CATEGORY of each category, in sorted order -- deterministic
    picked = {c: [] for c in CATEGORIES}
    for j in all_qa:
        c = categorise_qa_json(j)
        if c in picked and len(picked[c]) < N_PER_CATEGORY:
            picked[c].append(j)
        if all(len(v) >= N_PER_CATEGORY for v in picked.values()):
            break
    jsons_to_parse = [j for c in CATEGORIES for j in picked[c]]
    print()
    print('# paste into TEST_IDS above to pin this selection:')
    print('TEST_IDS = {')
    for c in CATEGORIES:
        ids = [os.path.basename(j).removesuffix('_qa.json') for j in picked[c]]
        print("    %-11s %r," % ("'%s':" % c, ids))
    print('}')

print()
print('testing on %d figures:' % len(jsons_to_parse))
for j in jsons_to_parse:
    print('   %-10s %s' % (categorise_qa_json(j), os.path.basename(j)))


total qa files with a matching image: 2001

testing on 150 figures:
   contour    vqa_000007_qa.json
   contour    vqa_000010_qa.json
   contour    vqa_000011_qa.json
   contour    vqa_000013_qa.json
   contour    vqa_000017_qa.json
   contour    vqa_000023_qa.json
   contour    vqa_000024_qa.json
   contour    vqa_000025_qa.json
   contour    vqa_000030_qa.json
   contour    vqa_000032_qa.json
   contour    vqa_000034_qa.json
   contour    vqa_000035_qa.json
   contour    vqa_000038_qa.json
   contour    vqa_000040_qa.json
   contour    vqa_000041_qa.json
   contour    vqa_000042_qa.json
   contour    vqa_000052_qa.json
   contour    vqa_000057_qa.json
   contour    vqa_000061_qa.json
   contour    vqa_000063_qa.json
   contour    vqa_000065_qa.json
   contour    vqa_000067_qa.json
   contour    vqa_000070_qa.json
   contour    vqa_000072_qa.json
   contour    vqa_000077_qa.json
   contour    vqa_000078_qa.json
   contour    vqa_000087_qa.json
   contour    vqa_000088_qa.json
   conto

In [6]:
# ==================== ARCHIVE AGING ====================
# Age every selected figure before it is sent, using the "archive-light"
# preset from page_aging.ipynb: decades in a box, then scanned, but gently.
#
# Each figure gets its own random draw -- which effects fire, the parameters
# each samples, and a 50/50 grayscale coin -- but the draw is seeded from the
# figure's OWN id, so the chatgpt, claude and gemini runs all see the identical
# aged image.  Comparing the three would otherwise be confounded by the aging
# rather than by the models.
#
# The aged images are written once into a shared directory; whichever notebook
# runs first creates them and the other two reuse them.
reload(utils.archive_aging)
from utils.archive_aging import build_aged_images, plan_for

# archive_base_seed and archive_gray_prob are set in the RUN CONFIG cell above
aged_imgs_dir = os.path.join(out_base, 'aged_imgs')
archive_manifest_path = os.path.join(out_base, 'archive_manifest.json')

selected_ids = [os.path.basename(j).removesuffix('_qa.json') for j in jsons_to_parse]

archive_manifest = build_aged_images(
    selected_ids, imgs_dir, aged_imgs_dir, img_format=img_format,
    base_seed=archive_base_seed, gray_prob=archive_gray_prob,
    preset=archive_preset, page_background=archive_page_background,
    manifest_path=archive_manifest_path)

# everything downstream reads images from here instead of the clean set
imgs_dir_original = imgs_dir
imgs_dir = aged_imgs_dir + os.sep
print('imgs_dir now:', imgs_dir)


preset     : archive-light  (white page background)
aged images: 0 written, 150 reused, 0 source images missing
  grayscale : 99 of 150 (66%)   [requested gray_prob=0.67]
  aged_dir  : /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/aged_imgs
  manifest  : /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/archive_manifest.json
imgs_dir now: /Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/aged_imgs/


In [7]:
def send_to_claude(question_list, client, image_path, encoded_image,
                    model="claude-haiku-4-5",
                    max_tokens=1000, temperature=0.1,
                    #media_type = 'image/', ## JPN take this out!
                    tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, fac=1.0, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, 
                    max_retries = 10, sleep_time=1, 
                    img_format='jpeg', reasoning=None, thinking = True, 
                    thinking_tokens=8000):
    """
    Sends the question to claude and collects response.clear

    system_prompt : if None, then defaults to the overall system prompt generated with the questions.
    """
    if system_prompt is None:
        system_prompt = question_list['persona']
    # update for caching
    system_prompt = [{
        "type": "text",
        "text": system_prompt,
        "cache_control": {"type": "ephemeral"}
    }]

    media_type = 'image/' + img_format #+= img_format
    #print('media type:', media_type)

    #print('system prompt:', system_prompt)

    iFac = 2.0
    success = False
    #['persona', 'context','question', 'format']
    attempt = 0
    while not success and attempt < max_retries:
        try:
            question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
            if reasoning is not None:
                question += " " + reasoning
            # lowercase the first word, just in case
            question = question.lstrip() # no whitespace
            question = question[0].lower() + question[1:]
            if verbose: print('   on question:',question)
            # Prepare the API request
            prompt = f"I am going to show you an image. Now, {question}"
            prompt_save = f"I am going to show you an image. Now, {question}"
            ##question_list['prompt'] = prompt
            
            if not test_run:
                if thinking and 'haiku' not in model:
                    # Send the request to the Claude api
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        #temperature=temperature,
                        thinking={"type": "adaptive"},  # <-- top-level parameter
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                elif thinking and 'haiku' in model:
                    # Send the request to the Claude api
                    if thinking_tokens is not None:
                        #thinking_tokens = max(min(thinking_tokens, max_tokens//2), 1024)
                        thinking_tokens = max(max_tokens-2000, 1024)
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        #temperature=temperature,
                        thinking={
                                "type": "enabled",
                                "budget_tokens": thinking_tokens   # min 1024, up to 64k for Haiku
                            },                        
                            messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                else:
                    # Send the request to the Claude api
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        # NOTE: no temperature.  anthropic>=1.0 removed the sampling
                        # knobs (temperature / top_p / top_k) from messages.create --
                        # Claude 4.6+ models do not accept them, and passing one is a
                        # TypeError, not an API error.  The low tier is therefore
                        # "default sampling, no thinking"; `temperature` is still
                        # accepted as an argument below so old calls don't break, but
                        # it is ignored.
                        #thinking={"type": "adaptive"},  # <-- top-level parameter
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                success = True
            else:
                success = True
        except (RateLimitError, APIConnectionError) as e:
            # retryable: 429 and transport failures
            if attempt < max_retries - 1:
                # Exponential backoff with jitter
                wait_time = sleep_time*(2 ** (attempt)) + np.random.uniform(0, 1)
                print(f"      {type(e).__name__}. Waiting {wait_time:.2f} seconds before retry {attempt + 1}")
                time.sleep(wait_time)
                attempt += 1
            else:
                print(f"Max retries exceeded. Error: {e}")
                raise
        except BadRequestError as e:
            # 400s are not retryable and are not fixed by shrinking the image --
            # record and move on rather than burning retries on them
            print(f"[ERROR] bad request (not retried): {e}")
            question_list['Error'] = 'BadRequest: ' + str(e)
            question_list['Response'] = ''
            question_list['Response String'] = ''
            question_list['raw answer'] = ''
            return question_list, '', system_prompt
        except APIStatusError as e:
            if e.status_code >= 500 and attempt < max_retries - 1:
                wait_time = sleep_time*(2 ** (attempt)) + np.random.uniform(0, 1)
                print(f"      server error {e.status_code}. Waiting {wait_time:.2f}s before retry {attempt + 1}")
                time.sleep(wait_time)
                attempt += 1
            else:
                raise
        except Exception as e:
            print('Exception!', str(e))
            new_fac = fac/iFac
            print('      new fac = ', new_fac)
            encoded_image = load_image(image_path,fac=new_fac, tmp_dir=tmp_dir)
            iFac += 1
            # lslkjfalsj
    
    #print(response)
    if not test_run:
        # Get the response from the API
        #answer = response.content[0].text #response.choices[0].message.content
        # try:
        #     answer = next(b.text for b in response.content if b.type == "text")
        # except:
        #     print(response.content)
        #     print([b.type for b in response.content])
        #     lsklsj
        text_blocks = [b for b in response.content if b.type == "text"]
        if text_blocks:
            answer = text_blocks[0].text
        else:
            print("WARNING: no text block found, content was:", response.content)
            print([b.type for b in response.content])
            answer = ""
            question_list['Error'] = 'No text block in response'
        question_list['raw answer'] = answer
        # also calculate usage
        usage = response.usage
        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
        success = True
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['raw answer'] = 'TEST RUN'

    return question_list, prompt_save, system_prompt


In [8]:
iMax = len(jsons_to_parse)   # quick test: just the selected examples
verbose = False
test_run = False # run w/o actually pinging openai
restart = False
#max_tokens=500
if max_tokens is None:
    max_tokens = 8000
#max_tokens=10000 
#max_tokens=1000 

# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
if reasoning_text is not None: # rephrease is reasoning is requested
    system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""

temperature=0.1
fac = 1.0
sleep = 5 # seconds
use_single_prompt = True # use 1 prompt and 1 image, if False will use multiple at a time

max_img_size = (8000,8000)

import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import get_img_json_pair

if test_run:
    sleep = 0.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    # imgs_dir points at the AGED copies, not the clean figures
    vqa_id = json_path.split('/')[-1].removesuffix('_qa.json')
    img_path = imgs_dir + vqa_id + '.' + img_format
    aging = archive_manifest.get(
        vqa_id, plan_for(vqa_id, base_seed=archive_base_seed,
                         gray_prob=archive_gray_prob,
                         preset=archive_preset,
                         page_background=archive_page_background))
    print(img_path)
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                      fac=fac, restart=restart, img_format=img_format,
                                                      tmp_dir=tmp_dir, max_img_size=max_img_size)
    if err:
        continue

    ###### create QA ########
    qa = []
    
    for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
        out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
               "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
               "reasoning":reasoning_text}
        qa.append(out)
    for level in ['Level 2', 'Level 3']:
        if level in base_json['VQA']:
            if 'Figure-level questions' in base_json['VQA'][level]:
                #print('** yes, level ***', level)
                for k,v in base_json['VQA'][level]['Figure-level questions'].items():
                    out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
                        "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
                        "reasoning":reasoning_text}
                    qa.append(out)
    
    # what kinds?
    #types = ['(words + list)', '(words)']
    types = []
    
    # get uniques
    level_parse = 'Level 1'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 2'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 3'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)

    responses = []; prompts = []; system_prompts = []
    if use_single_prompt:
        for question_list in qa:
            response, prompt, system_prompt_out = send_to_claude(question_list, client, img_path, encoded_image,
                        model = model, max_tokens=max_tokens, #media_type='image/' + img_format_media,
                        test_run = test_run, system_prompt=system_prompt, temperature=temperature, 
                        sleep_time=sleep, fac=fac, img_format=img_format, reasoning = reasoning_text, thinking=thinking)
            responses.append(response)
            question_list['prompt'] = prompt
            question_list['system prompt'] = system_prompt_out
        #import sys; sys.exit() # just do one
        time.sleep(sleep)
    time.sleep(sleep)

    # parse for errors
    #qa = parse_for_errors(qa, llm='claude')
    print('')
    print('**** Cleaned QA ****')
    #qa = parse_for_errors_claude(qa)
    qa = parse_for_errors(qa, llm='Claude')

    # record what was done to this image on every answer, so the aging
    # travels inside the pickle alongside the responses
    for question_list in qa:
        question_list['archive aging'] = aging

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model], ff)
        # ... and as a json beside it
        with open(dir_api + vqa_id + '_qa_archive.json', 'w') as ff:
            json.dump(aging, ff, indent=1)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()
print("!!!!!!!! DONE !!!!!!!!!!!")


on 0 of 150
/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/aged_imgs/vqa_000007.jpeg
   on question: assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
   on question: assume this is a figure made with matplotlib in Python. Examples of matplotlib colormaps are "rainbow" or "Reds". What is the colormap that was used in this figure? Please format the output as a json as {"colormap":""} to store the matplotlib colormap used in the figure. In addition to providing your answer, pl

## Look at data

Check out one, if you wanna:

In [9]:
pickles = glob(dir_api + '*.pickle')
pickles[:5]

['/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000146_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000014_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000128_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000035_qa.pickle',
 '/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000109_qa.pickle']

In [10]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [11]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images. Assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.',
 'A': {'plot style': 'fast'},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': {'explanation': "The figure uses a grayscale color scheme with a black background and white/gray contour patterns. The denote scale bar on the right side uses grayscale values from 2 to 20. The overall aesthetic matches matplotlib's 'grayscale' style, which is characterized by its monochromatic appearance, minimal colors, and focus on data visualization without colorful aesthetics. The black background and white gridlines/borders are typical of this style."},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': 'Assume this is a figure

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [12]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Claude A:', qa_pairs['raw answer'])
    print('')

/Users/jnaiman/astro_sky_image_vqa/LMM_outputs_n150_archive_light/claude_haiku/vqa_000146_qa.pickle
*********
Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
  Real A: {'plot style': 'fast'}
Claude A: ```json
{
  "plot_style": "grayscale"
}
```

```json
{
  "explanation": "The figure uses a grayscale color scheme with a black background and white/gray contour patterns. The denote scale bar on the right side uses grayscale values from 2 to 20. The

In [13]:
# how many have an error in thinking token limits
for pfile in pickles[ifile]:
    with open(pickles[ifile], 'rb') as f:
        qa_in = pickle.load(f)[0]
    for qa_pairs in qa_in:
        # question_list['Error']
        if 'Error' in qa_pairs:
            import sys; sys.exit()